<a href="https://colab.research.google.com/github/jmoussa99/football-ai/blob/main/testing_Praat_capabilities.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

# Praat speech analysis in Colab

This notebook extracts a short mono WAV from an audio or video file, transcribes it with CrisperWhisper, measures pitch, intensity, harmonicity and formants with [Parselmouth](https://parselmouth.readthedocs.io/en/stable/), estimates syllable peaks, and runs the original example emotion classifier.

Run cells in order. The default input is a short spoken-word [librosa example](https://librosa.org/doc/main/recordings.html), so **Run all** works without a personal Drive file. To analyze your own recording, set `INPUT_PATH` below to a file uploaded to `/content` or a path under `/content/drive/MyDrive`. A GPU runtime is recommended for transcription. The first model download may take several minutes.

In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "praat-parselmouth>=0.4.7,<0.5",
    "crisperwhisper[ct2,convert]",
    "librosa", "noisereduce", "pydub", "gdown",
])
print("Dependencies installed.")

## Choose the input

Leave `INPUT_PATH` empty to use the speech example. For a Colab upload, use the Files sidebar and set a path such as `/content/my_recording.mp4`. For a Drive file, use its full path under `/content/drive/MyDrive`; the cell mounts Drive only in that case. `MAX_DURATION_SECONDS` limits analysis and model runtime. Set it to `None` to process the complete file.

In [ ]:
from pathlib import Path
import subprocess

INPUT_PATH = ""  # e.g. "/content/drive/MyDrive/ESVA Video/my_video.mp4"
START_SECONDS = 0
MAX_DURATION_SECONDS = 30
ASR_MODEL_SIZE = "small"  # "medium", "turbo", or "large" also work

if INPUT_PATH:
    if INPUT_PATH.startswith("/content/drive/"):
        from google.colab import drive
        drive.mount("/content/drive")
    source_path = Path(INPUT_PATH).expanduser()
else:
    import librosa
    source_path = Path(librosa.ex("libri1"))

if not source_path.is_file():
    raise FileNotFoundError(f"Input file not found: {source_path}")
if START_SECONDS < 0 or (MAX_DURATION_SECONDS is not None and MAX_DURATION_SECONDS <= 0):
    raise ValueError("START_SECONDS must be >= 0 and MAX_DURATION_SECONDS must be positive or None")

output_dir = Path("/content/praat_outputs" if Path("/content").is_dir() else "praat_outputs")
output_dir.mkdir(parents=True, exist_ok=True)
audio_path = output_dir / "analysis_audio.wav"

command = ["ffmpeg", "-y", "-nostdin", "-loglevel", "error", "-ss", str(START_SECONDS), "-i", str(source_path)]
if MAX_DURATION_SECONDS is not None:
    command += ["-t", str(MAX_DURATION_SECONDS)]
command += ["-vn", "-acodec", "pcm_s16le", "-ar", "16000", "-ac", "1", str(audio_path)]
subprocess.run(command, check=True)
if not audio_path.is_file() or audio_path.stat().st_size <= 44:
    raise RuntimeError("No audio was extracted. Check the input file and start time.")

print(f"Input: {source_path}")
print(f"16 kHz mono WAV: {audio_path}")

## Transcription

[CrisperWhisper](https://github.com/nyrahealth/CrisperWhisper/blob/main/DOCS.md) accepts the extracted WAV directly. The ASR model stays in `asr_model`; the emotion model later has a separate name.

In [ ]:
import re
from crisperwhisper import CrisperWhisperModel

asr_model = CrisperWhisperModel(ASR_MODEL_SIZE)
result = asr_model.transcribe(str(audio_path), language="en", mode="verbatim")
continuous_text = result.text.strip()
sentences = [part.strip() for part in re.split(r"(?<=[.!?])\s+", continuous_text) if part.strip()]
transcript = "\n\n".join(
    " ".join(sentences[index:index + 5])
    for index in range(0, len(sentences), 5)
)
transcript_path = output_dir / "transcript.txt"
transcript_path.write_text(transcript, encoding="utf-8")
print(transcript or "No speech was recognized in this clip.")
print(f"\nSaved: {transcript_path}")

## Praat measurements

Pitch relates to vocal-fold vibration; intensity approximates loudness; harmonic-to-noise ratio describes voice quality; and formants describe resonances of the vocal tract. Unvoiced or undefined measurements appear as gaps in the plots.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import parselmouth

sound = parselmouth.Sound(str(audio_path))
hop_dur = 0.01
pitch = sound.to_pitch(time_step=hop_dur)
harmonicity = sound.to_harmonicity(time_step=hop_dur)
formants = sound.to_formant_burg(
    time_step=hop_dur,
    max_number_of_formants=3,
    maximum_formant=4500,
    window_length=0.005,
    pre_emphasis_from=50.0,
)
intensity = sound.to_intensity(minimum_pitch=75.0, time_step=hop_dur, subtract_mean=False)
spectrogram = sound.to_spectrogram(window_length=0.04)

times = pitch.ts()
if len(times) == 0:
    raise ValueError("The extracted clip is too short for Praat analysis.")
pitch_vals = np.array([pitch.get_value_at_time(t) for t in times])
harmonicity_vals = np.array([harmonicity.get_value(t) for t in times])
intensity_vals = np.array([intensity.get_value(t) for t in times])
formant_vals = [
    np.array([formants.get_value_at_time(number, t) for t in times])
    for number in (1, 2, 3)
]

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
for ax, values, label in zip(
    axes.flat[:3],
    (pitch_vals, intensity_vals, harmonicity_vals),
    ("Pitch (Hz)", "Intensity (dB)", "Harmonic-to-noise ratio (dB)"),
):
    ax.plot(times, values)
    ax.set_ylabel(label)
    ax.grid(True)
for number, values in enumerate(formant_vals, 1):
    axes[1, 1].plot(times, values / 1000, label=f"F{number}")
axes[1, 1].set_ylabel("Formant frequency (kHz)")
axes[1, 1].legend()
axes[1, 1].grid(True)
for ax in axes[1]:
    ax.set_xlabel("Time (s)")
fig.tight_layout()
plt.show()

power_db = 10 * np.log10(np.maximum(spectrogram.values, np.finfo(float).tiny))
fig, ax = plt.subplots(figsize=(12, 5))
image = ax.pcolormesh(
    spectrogram.x_grid(), spectrogram.y_grid(), power_db,
    vmin=power_db.max() - 70, vmax=power_db.max(), cmap="RdBu_r", shading="auto",
)
fig.colorbar(image, ax=ax, label="Power (dB)")
ax.set(xlabel="Time (s)", ylabel="Frequency (Hz)", ylim=(0, 4500))
fig.tight_layout()
plt.show()

## Approximate syllable peaks

This is an intensity-peak heuristic, not a full implementation of the de Jong and Wempe Praat script. It keeps prominent peaks within 25 dB of the clip's maximum and at least 0.12 seconds apart, then filters for voiced frames. Background speech and noise can affect the count.

In [ ]:
import pandas as pd
from scipy.signal import find_peaks

intensity_times = intensity.xs()
intensity_db = intensity.values[0]
finite_db = intensity_db[np.isfinite(intensity_db)]
if len(finite_db) == 0 or len(intensity_times) < 2:
    syllable_times = np.array([])
else:
    frame_step = float(np.median(np.diff(intensity_times)))
    candidates, _ = find_peaks(
        intensity_db,
        prominence=2,
        distance=max(1, round(0.12 / frame_step)),
    )
    threshold = finite_db.max() - 25
    syllable_times = np.array([
        intensity_times[index] for index in candidates
        if intensity_db[index] >= threshold
        and np.isfinite(pitch.get_value_at_time(float(intensity_times[index])))
    ])

syllables = pd.DataFrame({"time_seconds": syllable_times})
print(f"Approximate syllable peaks: {len(syllables)}")
display(syllables.head(20))

## Example emotion classifier

The original notebook downloads a small pretrained model from public Google Drive files. Its output is an illustrative model prediction, not a validated assessment of a speaker's emotional state. This cell uses the same extracted audio as the rest of the notebook and analyzes its first roughly seven seconds. The model and weights must remain publicly available for this section to run.

In [ ]:
import json
import gdown
import tensorflow as tf

model_json_path = output_dir / "model8723.json"
model_weights_path = output_dir / "model8723_weights.h5"
for file_id, path in (
    ("1-0EzrxwKruKRplwiXvFbSn9mz7n3r2Jy", model_json_path),
    ("1-2k-0_TrwUj9c4ccQCyzYRFosDtXBpIl", model_weights_path),
):
    if not path.is_file():
        downloaded = gdown.download(id=file_id, output=str(path), quiet=False)
        if not downloaded or not path.is_file():
            raise RuntimeError(f"Could not download the public emotion model file: {path.name}")

model_config = json.loads(model_json_path.read_text())
for layer in model_config.get("config", {}).get("layers", []):
    layer.get("config", {}).pop("time_major", None)

custom_objects = {
    name: getattr(tf.keras.layers, name)
    for name in ("InputLayer", "LSTM", "Dense", "Dropout", "Activation", "Flatten", "Bidirectional", "TimeDistributed")
}
emotion_model = tf.keras.models.model_from_json(json.dumps(model_config), custom_objects=custom_objects)
emotion_model.load_weights(str(model_weights_path))
emotion_model.summary()

In [ ]:
import librosa
import noisereduce as nr
from pydub import AudioSegment, effects

def preprocess_emotion(file_path, frame_length=2048, hop_length=512, frames=339):
    segment = effects.normalize(AudioSegment.from_wav(str(file_path)), headroom=5.0)
    sample_rate = segment.frame_rate
    samples = np.array(segment.get_array_of_samples(), dtype=np.float32)
    clean = nr.reduce_noise(y=samples, sr=sample_rate)
    features = np.concatenate((
        librosa.feature.rms(y=clean, frame_length=frame_length, hop_length=hop_length).T,
        librosa.feature.zero_crossing_rate(clean, frame_length=frame_length, hop_length=hop_length).T,
        librosa.feature.mfcc(y=clean, sr=sample_rate, n_mfcc=13, hop_length=hop_length).T,
    ), axis=1)
    features = features[:frames]
    if len(features) < frames:
        features = np.pad(features, ((0, frames - len(features)), (0, 0)))
    return np.nan_to_num(features, copy=False).astype(np.float32)[None, :, :]

emotions = ["neutral", "calm", "happy", "sad", "angry", "fearful", "disgust", "surprised"]
probabilities = emotion_model.predict(preprocess_emotion(audio_path), verbose=0)[0]
fig, ax = plt.subplots(figsize=(10, 3))
ax.bar(emotions, probabilities, color="darkturquoise")
ax.set(ylabel="Model probability", title=f"Top prediction: {emotions[int(np.argmax(probabilities))]}")
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
plt.show()